# Fine-Tune Whisper on the Bisaya Speech Corpus (Kaggle Notebook)

This is a Kaggle-notebook adaptation of Hugging Face's
[Fine-Tune Whisper for Multilingual ASR](https://huggingface.co/blog/fine-tune-whisper)
tutorial, pointed at this project's own Bisaya (Cebuano) speech corpus
instead of the tutorial's original Hindi/Common Voice example. It trains
a third ASR system for this project's benchmark, alongside the Kaldi
HMM-GMM model (`train_kaldi.ipynb`) and ElevenLabs Scribe
(`evaluate_elevenlabs.ipynb`) -- see the repo's `README.md` for the rest
of that pipeline. This notebook is standalone: it doesn't feed into
`compare.ipynb` automatically.

**Differences from the original Colab tutorial**, besides the dataset:
- GPU/internet setup, Hugging Face auth, and long-run persistence all use
  Kaggle's own mechanisms instead of Colab's (Section "Kaggle Setup" below).
- The corpus has to be **uploaded to Kaggle as a Dataset first** (Section
  "Upload the Corpus to Kaggle" below) -- Kaggle notebooks can't read your
  local filesystem the way Colab can read from Google Drive or a direct
  download.
- The train/test split reuses the exact same speaker-independent split
  (by speaker, ~80/20, fixed seed 42) that `train_kaldi.ipynb` uses, so
  Whisper's held-out test speakers match Kaldi's -- a prerequisite for any
  later three-way comparison.
- Whisper has no dedicated Cebuano/Bisaya language token (~99 languages
  are supported; Bisaya isn't one). This notebook uses `"tl"` (Tagalog --
  Whisper's closest Philippine-language code) for the tokenizer/
  generation language conditioning -- a real approximation, not an exact
  match, called out again where it's used.

**Fixed after a collapsed-generation run** (repeated `<|fr|>` tokens, 100%
WER): the decoder prompt is now forced deterministically via
`forced_decoder_ids` instead of relying on generate()-time language
auto-detection, and the learning rate is back to a full-fine-tuning-scale
`1e-5` -- the notebook installs `peft` and named its output dir
`-lora`, but never actually wrapped the model in a LoRA adapter, so it was
silently doing full-parameter fine-tuning at a LoRA-scale (100x too high)
learning rate. See the note on the training-configuration cell below if
you want to add real LoRA instead of reverting the learning rate.

## Kaggle Setup

1. **Enable a GPU**: notebook Settings (right sidebar) -> Accelerator ->
   GPU T4 x2 (or P100/other, if available). Kaggle currently grants a
   weekly GPU-hour quota per account -- check your remaining quota in
   Settings before starting a long run.
2. **Enable internet access**: Settings -> Internet -> On. Required for
   `pip install`, downloading the pretrained Whisper checkpoint, and
   pushing to the Hugging Face Hub.
3. **Attach the corpus dataset**: see "Upload the Corpus to Kaggle" below
   -- do this once, then attach it via Add Input on every notebook that
   needs it.
4. **Add your Hugging Face token as a Kaggle Secret**: Add-ons -> Secrets
   -> add a secret named `HF_TOKEN` with a Hugging Face
   [write access token](https://huggingface.co/settings/tokens) as the
   value. Used in "Hugging Face Authentication" below instead of the
   interactive `notebook_login()` widget, so the notebook can run
   unattended (Save & Run All).

## Upload the Corpus to Kaggle

One-time step, done outside this notebook, before it can run:

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) -> **New
   Dataset**.
2. Upload every file under this project's `data/bisaya_audio/` (the
   Parquet shards) -- drag-and-drop in the browser, or use the
   [Kaggle API](https://www.kaggle.com/docs/api) from the machine that
   has the corpus locally:
   ```bash
   pip install kaggle
   # ~/.kaggle/kaggle.json holds your API credentials (Kaggle account ->
   # Settings -> Create New Token)
   kaggle datasets init -p data/bisaya_audio
   # edit the generated dataset-metadata.json: set a title/id, e.g.
   #   "id": "your-kaggle-username/bisaya-audio-corpus"
   kaggle datasets create -p data/bisaya_audio
   ```
3. In this notebook (or any Kaggle notebook that needs the corpus): **Add
   Input** (right sidebar) -> search for the dataset you just created ->
   Add. It appears under `/kaggle/input/<dataset-slug>/`.
4. Set `CORPUS_DIR` in the next cell to match wherever your Parquet files
   actually land under `/kaggle/input/`.

This corpus is not public -- keep the uploaded Kaggle Dataset **Private**
unless you have the right to publish it.

## Prepare Environment

In [ ]:
!nvidia-smi


In [ ]:
pip install -q --force-reinstall "numpy<2.0.0" "scipy<1.14.0" "protobuf>=5.29.1,<6.0.0" pyarrow datasets transformers accelerate peft librosa soundfile evaluate jiwer


In [ ]:
import os

print("Environment setup complete! Restarting kernel...")
os._exit(0)


In [ ]:
import os, subprocess

print("Installing system FFmpeg and setting audio decoding backend...")

# 1. Install system FFmpeg libraries required by C-extension audio decoders
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg",
    shell=True,
    check=True,
)

# 2. Disable torchcodec explicitly so datasets uses soundfile/librosa
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"

print("FFmpeg installed and torchcodec disabled!")


### Hugging Face Authentication

Uses the `HF_TOKEN` Kaggle Secret set up above, rather than the
interactive `notebook_login()` widget the original Colab tutorial uses --
this keeps the notebook runnable unattended via Kaggle's **Save & Run
All (Commit)**.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# Login and expose as environment variable for all Hugging Face libraries
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token


## Load Dataset

Reads every Parquet shard in `CORPUS_DIR` via 🤗 `datasets` directly --
the corpus's `audio` column is already stored in `datasets`' own Audio
struct format (`{bytes, path}`, with the feature type recorded in the
Parquet file's own schema metadata), so `load_dataset("parquet", ...)`
decodes it automatically, the same way it would for a published HF
dataset like the tutorial's Common Voice.

In [ ]:
from pathlib import Path

# Adjust to match the slug/path your attached Kaggle Dataset actually uses.
CORPUS_DIR = Path("/kaggle/input/datasets/troymerales/bisaya-audio")

parquet_files = sorted(str(p) for p in CORPUS_DIR.glob("*.parquet"))
assert parquet_files, f"No Parquet files found under {CORPUS_DIR} -- check the dataset is attached (Add Input)."
print(f"Found {len(parquet_files)} Parquet shard(s)")


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("parquet", data_files=parquet_files, split="train")
print(raw_dataset)


### Speaker-Independent Train/Test Split

Same split `train_kaldi.ipynb` Section 8 uses -- by speaker (not
utterance), ~80/20, fixed seed 42 -- so this notebook's held-out test
speakers are the same ones Kaldi never trained on. Reused verbatim rather
than re-derived, so the split stays identical if this cell or
`train_kaldi.ipynb`'s changes independently.

`test_fraction` is back to `0.2` here (was `0.1`) -- on a ~77-utterance
corpus a 10% test split can leave only one or two speakers held out,
which makes eval WER extremely high-variance (one bad utterance can swing
it by tens of points) and breaks parity with `train_kaldi.ipynb`'s own
80/20 split.

In [ ]:
from datasets import DatasetDict
import pandas as pd

SPLIT_SEED = 42


def speaker_independent_split(speakers, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(speakers)
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])
    assert train_speakers.isdisjoint(test_speakers)
    return train_speakers, test_speakers


all_speakers = set(raw_dataset.unique("speaker_id"))
train_speakers, test_speakers = speaker_independent_split(all_speakers)

bisaya = DatasetDict(
    {
        "train": raw_dataset.filter(lambda ex: ex["speaker_id"] in train_speakers),
        "test": raw_dataset.filter(lambda ex: ex["speaker_id"] in test_speakers),
    }
)

print(f"seed = {SPLIT_SEED}")
print(f"train: {len(train_speakers)} speakers, {len(bisaya['train'])} audio files")
print(f"test:  {len(test_speakers)} speakers, {len(bisaya['test'])} audio files")

# On a very small corpus a couple of unlucky speakers can leave a split
# empty -- fail loudly here instead of getting a meaningless WER (or a
# crash inside evaluate's wer metric) much later in the notebook.
assert len(bisaya["train"]) > 0, "train split is empty -- check speaker_id values in the corpus"
assert len(bisaya["test"]) > 0, "test split is empty -- too few speakers for this test_fraction"


In [ ]:
# Keep audio and transcript columns
keep_cols = {"audio", "transcript"}
bisaya = bisaya.remove_columns(
    [c for c in bisaya["train"].column_names if c not in keep_cols]
)
print("Columns after pruning:", bisaya["train"].column_names)
# Output should be: ['audio', 'transcript']


In [ ]:
# Drop rows with a missing/empty transcript or missing/empty audio bytes
# before doing any real work -- cheap structural checks (no audio decode),
# so this runs fast even though it's a full pass over the corpus.
def _is_valid_row(example):
    transcript = (example.get("transcript") or "").strip()
    audio = example.get("audio")
    has_audio = isinstance(audio, dict) and bool(audio.get("bytes"))
    return bool(transcript) and has_audio


before_counts = {split: len(bisaya[split]) for split in bisaya}
bisaya = bisaya.filter(_is_valid_row, num_proc=1)
after_counts = {split: len(bisaya[split]) for split in bisaya}
print(f"null/corrupted rows dropped: {before_counts} -> {after_counts}")

assert len(bisaya["train"]) > 0, "no valid training rows left after filtering"
assert len(bisaya["test"]) > 0, "no valid test rows left after filtering -- eval/WER will be meaningless"


## Prepare Feature Extractor, Tokenizer and Data

### Load WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

`language="tl"` (Tagalog) is Whisper's closest built-in Philippine-language
code -- there is no Cebuano/Bisaya entry in Whisper's ~99-language list.
This is an approximation the model was not specifically designed for;
treat any language-conditioning benefit from it as best-effort, not exact.
The short ISO code is used instead of the full name `"Tagalog"` so it's
unambiguous everywhere it's reused below (tokenizer, processor, and the
forced decoder prompt).

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

MODEL_CHECKPOINT = "openai/whisper-small"
LANGUAGE = "tl"  # ISO code for Tagalog -- Whisper's closest built-in language to Bisaya/Cebuano (see note above)

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")

# Sanity-check that the tokenizer is actually prepending Whisper's mandatory
# prefix sequence to every label -- this is what language=/task= above does
# implicitly via set_prefix_tokens(), and prepare_dataset() below relies on
# it. If this ever fails, labels won't start where generation expects.
expected_prefix = ["<|startoftranscript|>", f"<|{LANGUAGE}|>", "<|transcribe|>", "<|notimestamps|>"]
actual_prefix = tokenizer.convert_ids_to_tokens(tokenizer.prefix_tokens)
assert actual_prefix == expected_prefix, (
    f"tokenizer prefix tokens are {actual_prefix}, expected {expected_prefix}"
)
print("decoder prefix tokens:", actual_prefix)


### Prepare Data

In [ ]:
print(bisaya["train"][0]["transcript"])


In [ ]:
print(bisaya.column_names)


In [ ]:
from datasets import Audio

# 1. Ensure 16kHz audio sampling
bisaya = bisaya.cast_column("audio", Audio(sampling_rate=16000))


# 2. Extract Mel-spectrograms & truncate labels safely to 448
def prepare_dataset(batch):
    audio = batch["audio"]
    array = audio["array"]
    sr = audio["sampling_rate"]

    # Truncate audio array to 30 seconds max (480,000 samples at 16kHz)
    max_samples = 30 * sr
    if len(array) > max_samples:
        array = array[:max_samples]

    # Extract log-Mel features
    batch["input_features"] = feature_extractor(
        array, sampling_rate=sr
    ).input_features[0]

    # tokenizer(...) already prepends the mandatory prefix sequence
    # (<|startoftranscript|>, <|tl|>, <|transcribe|>, <|notimestamps|>) and
    # appends <|endoftext|> -- set by WhisperTokenizer.from_pretrained's
    # language=/task= args above (verified there by assertion). Truncate to
    # Whisper's max decoder position (448) as a hard safety cap.
    batch["labels"] = tokenizer(
        batch["transcript"], max_length=448, truncation=True
    ).input_ids

    return batch


# 3. Drop all current columns automatically (prevents KeyError/ValueError)
# num_proc=1 -- Kaggle's multiprocessing + datasets.map() combination
# reliably hangs/freezes with num_proc>1 in this environment.
bisaya = bisaya.map(
    prepare_dataset,
    remove_columns=bisaya["train"].column_names,
    num_proc=1,
)

print("Processed dataset columns:", bisaya["train"].column_names)
# Expected output: ['input_features', 'labels']


In [ ]:
# Size generation_max_length off the actual tokenized labels instead of
# the fixed 225 guess -- used by training_args below, capped at Whisper's
# decoder position limit (448) that prepare_dataset() already truncates to.
max_label_len = max(
    max(len(x) for x in bisaya["train"]["labels"]),
    max(len(x) for x in bisaya["test"]["labels"]),
)
GENERATION_MAX_LENGTH = min(448, max_label_len + 10)
print(f"longest tokenized label = {max_label_len} tokens -> generation_max_length = {GENERATION_MAX_LENGTH}")


## Training and Evaluation

Same 🤗 Trainer-based pipeline as the original tutorial: load a
pretrained checkpoint, define a data collator, define the WER metric,
configure and run training.

### Load a Pre-Trained Checkpoint

In [ ]:
from transformers import WhisperForConditionalGeneration
import torch
import gc

model = WhisperForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)

# Deterministically force the first 4 decoder tokens
# (<|startoftranscript|>, <|tl|>, <|transcribe|>, <|notimestamps|>) instead
# of leaving generate() to auto-detect language from the encoder output.
# On a small fine-tuned model that auto-detection is unreliable -- it's the
# direct cause of the reported failure mode: generation collapsing into a
# repeated wrong-language special token (e.g. <|fr|>), which decodes to an
# empty string and scores 100% WER. This is saved into the model's
# generation_config.json by save_pretrained()/push_to_hub(), so it also
# applies automatically to any later pipeline()/deploy inference -- no need
# to pass forced_decoder_ids by hand at generate()-time anymore.
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task="transcribe"
)

torch.cuda.empty_cache()
gc.collect()


### Define a Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad pre-extracted log-Mel input_features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label token sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding tokens in labels with -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Strip initial decoder start token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


### Evaluation Metrics

In [ ]:
import evaluate
import numpy as np

# Load Word Error Rate metric
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 in labels with pad_token_id so decoding doesn't crash
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Guard against a still-collapsed model whose predictions decode to ""
    # (all special tokens) -- evaluate's wer metric divides by reference
    # length and raises on an empty reference, which would otherwise crash
    # trainer.evaluate() mid-training instead of just reporting a bad WER.
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 100.0}
    preds, labels = zip(*pairs)
    wer = 100 * metric.compute(predictions=list(preds), references=list(labels))
    return {"wer": wer}


### Define the Training Configuration

**Learning rate fixed: `1e-3` -> `1e-5`.** The pip-install cell installs
`peft` and `output_dir` was named `...-lora`, but no `LoraConfig`/
`get_peft_model()` call ever wraps the model anywhere in this notebook --
so training was doing full-parameter fine-tuning at a LoRA-scale learning
rate (100x too high for full fine-tuning). On a small pretrained
transformer that's a direct route to the reported collapse: weights blow
up within the first few steps, and generation degenerates into repeating
a single token. `1e-5` matches the original tutorial's full-fine-tuning
rate. (If you actually want LoRA -- a reasonable choice for a ~77-example
corpus, since it trains far fewer parameters and overfits less -- that's
a separate, deliberate change: wrap `model` with
`peft.get_peft_model(model, LoraConfig(...))` and call
`model.enable_input_require_grads()` before building the Trainer, then
the `1e-3` rate becomes appropriate again. Left out here since it wasn't
part of the current code and changes the training approach, not just
fixes a bug in it.)

`num_train_epochs=5` / `eval_strategy="epoch"` / `save_strategy="epoch"`
were already correctly sized for this small corpus -- no change needed
there.

`push_to_hub=True` is still the main persistence strategy on Kaggle, same
as the original tutorial recommends for Colab: Kaggle notebook sessions
are also ephemeral (working-directory contents don't survive past the
session unless explicitly saved), so periodic checkpoints pushed to the
Hub protect the run against an interrupted or killed session.

In [ ]:
print("Train dataset column names:", bisaya["train"].column_names)
print("First item keys:", bisaya["train"][0].keys())


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bisaya",  # was "...-lora" -- no LoRA adapter is actually applied, see note above
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size = 16
    learning_rate=1e-5,  # was 1e-3 (LoRA-scale) -- see note above, this is the fix for the reported collapse
    warmup_ratio=0.1,  # Uses 10% of total steps for warmup instead of a fixed 500 steps
    num_train_epochs=5,  # Loops through the ~3.14h train set 5 times (~75 total steps)
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",  # Evaluates WER once at the end of every epoch
    save_strategy="epoch",  # Saves a checkpoint once at the end of every epoch
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,  # computed from the actual tokenized labels, not guessed
    generation_num_beams=1,
    logging_steps=5,  # Reduced logging interval since total steps will be < 100
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=bisaya["train"],
    eval_dataset=bisaya["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    # Stops the run once eval WER hasn't improved for 3 epochs, instead of
    # burning the rest of num_train_epochs after the model has collapsed.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


### Training

**Running unattended on Kaggle:** unlike Colab (which needs the browser
tab open and a JS keep-alive trick to survive idle disconnects), Kaggle
runs a notebook as a real background batch job when you click **Save
Version -> Save & Run All (Commit)** -- you can close the browser and it
keeps training. Use interactive "Edit" mode only for iterating on the
early cells; switch to a commit run for the actual training job.

Watch your GPU-hour quota -- a run that hits Kaggle's session time limit
partway through is still recoverable via `push_to_hub`'s periodic
checkpoints (re-load with `from_pretrained` on your Hub repo and resume),
but budgeting the run to fit inside one session avoids that entirely.

In [ ]:
# Launch fine-tuning run
trainer.train()


In [ ]:
import torch
import gc
import pandas as pd

# Confirm the best checkpoint (already loaded via load_best_model_at_end)
# is actually usable BEFORE pushing anything to the Hub.
eval_metrics = trainer.evaluate()
print("Evaluation Results:", eval_metrics)
if eval_metrics.get("eval_wer", 100) >= 100:
    print(
        "WARNING: eval WER is 100% -- the model likely collapsed. "
        "Review the learning-rate/forced_decoder_ids fixes above before "
        "pushing to the Hub."
    )

# Side-by-side sample check -- forced_decoder_ids is now set globally on
# model.generation_config (see the model-loading cell), so plain
# model.generate() below already uses it; no per-call override needed.
sample_batch = bisaya["test"].select(range(min(5, len(bisaya["test"]))))
input_features = torch.stack(
    [torch.tensor(ex["input_features"]) for ex in sample_batch]
).to(model.device)

with torch.no_grad():
    generated_ids = model.generate(input_features, max_new_tokens=GENERATION_MAX_LENGTH)

pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
label_ids = [
    [t if t != -100 else tokenizer.pad_token_id for t in ex["labels"]]
    for ex in sample_batch
]
ref_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame({"Reference (Ground Truth)": ref_texts, "Model Prediction": pred_texts}))

torch.cuda.empty_cache()
gc.collect()


Update `kwargs` to match your run, then push the model card + final
checkpoint to the Hub. `dataset_tags`/`dataset_args` are omitted since
this corpus isn't a public Hub dataset (see "Upload the Corpus to
Kaggle" above) -- don't set them to a dataset id that doesn't exist.

In [ ]:
kwargs = {
    "dataset": "Bisaya Speech Corpus (private)",
    "language": "ceb",  # ISO 639-3 for Cebuano/Bisaya -- Hub model-card metadata only;
                         # unrelated to Whisper's own "tl" language token used above
    "model_name": "Whisper Small Bisaya",  # a 'pretty' name for our model
    "finetuned_from": MODEL_CHECKPOINT,
    "tasks": "automatic-speech-recognition",
}


In [ ]:
trainer.push_to_hub(**kwargs)


# Export for local machine

In [ ]:
# Save the model and processor explicitly from the trainer
save_directory = "./whisper_bisaya_final"

trainer.save_model(save_directory)
processor.save_pretrained(save_directory)

print("Saved model and processor to:", save_directory)


In [ ]:
import shutil
from IPython.display import FileLink

# 1. Zip the output folder
shutil.make_archive("whisper_bisaya_model", "zip", save_directory)

# 2. Display the download link
print("Click below to download your fine-tuned Whisper model:")
FileLink(r"whisper_bisaya_model.zip")


## Closing Remarks

This notebook fine-tunes Whisper small on this project's Bisaya corpus
using 🤗 Datasets, Transformers, and the Hugging Face Hub, adapted for
Kaggle's environment (GPU/internet setup, Kaggle Secrets for
authentication, dataset upload, and unattended long-run training via
Save & Run All). See the original
[fine-tuning blog post](https://huggingface.co/blog/fine-tune-whisper)
for the underlying theory, and this project's own `README.md`/`CLAUDE.md`
for how this fits alongside the Kaldi and ElevenLabs systems in the wider
benchmark.